# 01 — Foundations: Scalars, Vectors, Matrices & Tensors

This notebook builds up the mathematical language you need before touching tensor networks.
We start from the very basics and work towards multi-dimensional arrays (tensors), index notation, and key operations.

---

## 1. The Hierarchy of Arrays

| Name | Order | Example shape |
|------|-------|---------------|
| Scalar | 0 | `()` |
| Vector | 1 | `(n,)` |
| Matrix | 2 | `(m, n)` |
| Tensor | ≥ 3 | `(d1, d2, d3, ...)` |

A **tensor** is simply a multi-dimensional array of numbers. The number of indices it has is called its **order** (or rank).  
Each index runs over a **mode** (a.k.a. dimension or way).

In Einstein / index notation, a rank-3 tensor is written:

$$T_{ijk}, \quad i \in [I],\; j \in [J],\; k \in [K]$$

In [ ]:
import numpy as np

# --- Scalar ---
s = 3.14
print(f"Scalar:  value={s}, ndim={np.ndim(s)}, shape={np.shape(s)}")

# --- Vector ---
v = np.array([1.0, 2.0, 3.0])
print(f"Vector:  value={v}, ndim={v.ndim}, shape={v.shape}")

# --- Matrix ---
M = np.array([[1, 2, 3],
              [4, 5, 6]])
print(f"Matrix:\n{M}\n  ndim={M.ndim}, shape={M.shape}")

# --- Rank-3 Tensor ---
T = np.arange(24).reshape(2, 3, 4)   # shape (I=2, J=3, K=4)
print(f"\nRank-3 Tensor shape={T.shape}, ndim={T.ndim}")
print("T[0] (first frontal slice):\n", T[0])

## 2. Accessing Elements — Index Notation

For a tensor $T$ of shape $(I, J, K)$:

$$T_{ijk} \equiv \texttt{T[i, j, k]}$$

Think of the tensor as a 3-D array of numbers.  
Reading element $(i,j,k)$ means: go to page $i$, row $j$, column $k$.

In [ ]:
# Element T_{0,1,2} — page 0, row 1, column 2
elem = T[0, 1, 2]
print(f"T[0,1,2] = {elem}")

# Fibre along mode-2 (all k, fixed i=0, j=1)
fibre = T[0, 1, :]  # equivalent to T_{0,1,k} for all k
print(f"Mode-2 fibre T[0,1,:] = {fibre}")

# Slice: all i,j at fixed k=0  --> a matrix
frontal_slice = T[:, :, 0]
print(f"Frontal slice T[:,:,0]:\n{frontal_slice}")

## 3. Key Operations

### 3.1 Outer Product

Given vectors $\mathbf{a} \in \mathbb{R}^I$ and $\mathbf{b} \in \mathbb{R}^J$:

$$C_{ij} = a_i \, b_j \quad \Leftrightarrow \quad \mathbf{C} = \mathbf{a} \otimes \mathbf{b}$$

This generalises to higher orders:

$$T_{ijk} = a_i \, b_j \, c_k$$

Such a tensor is called a **rank-1 tensor** (one outer product of vectors).

In [ ]:
a = np.array([1.0, 2.0, 3.0])
b = np.array([4.0, 5.0])
c = np.array([6.0, 7.0, 8.0, 9.0])

# Rank-1 tensor via outer products
T_rank1 = np.einsum('i,j,k->ijk', a, b, c)   # shape (3, 2, 4)
print("Rank-1 tensor shape:", T_rank1.shape)
print("T_rank1[0,0,0] should equal a[0]*b[0]*c[0] =", a[0]*b[0]*c[0])
print("T_rank1[0,0,0] =", T_rank1[0,0,0])

### 3.2 Inner / Dot Product

$$\langle \mathbf{u}, \mathbf{v} \rangle = \sum_i u_i v_i$$

For matrices, the standard matrix product is:

$$C_{ik} = \sum_j A_{ij} B_{jk}$$

This is a **contraction** over the shared index $j$.

In [ ]:
u = np.array([1.0, 2.0, 3.0])
v = np.array([4.0, 5.0, 6.0])
dot = np.dot(u, v)
print(f"dot(u,v) = {dot}")

A = np.random.randn(3, 4)
B = np.random.randn(4, 5)
C = A @ B                        # matrix multiply = contract over index 1 of A and index 0 of B
print(f"A @ B shape: {C.shape}")

# same via einsum — notice 'j' is the contracted (summed) index
C_ein = np.einsum('ij,jk->ik', A, B)
print("Results match:", np.allclose(C, C_ein))

### 3.3 Tensor Contraction (generalised)

Contracting two tensors over a shared index is the core operation in tensor networks.

$$D_{imn} = \sum_j A_{ij} B_{jmn}$$

The shared index $j$ is **summed out** (contracted). The resulting tensor retains all free indices.

In [ ]:
A = np.random.randn(3, 4)         # shape (i=3, j=4)
B = np.random.randn(4, 5, 6)     # shape (j=4, m=5, n=6)

# Contract index j
D = np.einsum('ij,jmn->imn', A, B)
print(f"D shape: {D.shape}")  # should be (3, 5, 6)

# Verify one element manually
d_000 = sum(A[0, j] * B[j, 0, 0] for j in range(4))
print(f"Manual D[0,0,0] = {d_000:.6f}")
print(f"einsum D[0,0,0] = {D[0,0,0]:.6f}")
print("Match:", np.isclose(d_000, D[0,0,0]))

### 3.4 Mode-n Unfolding (Matricisation)

To apply linear algebra tools to tensors, we often **unfold** them into a matrix.

The **mode-$n$ unfolding** $T_{(n)}$ rearranges $T$ so that mode $n$ becomes the rows and all other modes are flattened into columns.

For $T$ of shape $(I, J, K)$:

$$T_{(1)} \in \mathbb{R}^{I \times JK}, \quad T_{(2)} \in \mathbb{R}^{J \times IK}, \quad T_{(3)} \in \mathbb{R}^{K \times IJ}$$

In [ ]:
import tensorly as tl

T = np.arange(24).reshape(2, 3, 4).astype(float)   # shape (2, 3, 4)
print("Original T shape:", T.shape)

for mode in range(3):
    unfolded = tl.unfold(T, mode)
    print(f"Mode-{mode} unfolding shape: {unfolded.shape}")

# Fold back to original
T0_unfolded = tl.unfold(T, 0)
T_recon = tl.fold(T0_unfolded, 0, T.shape)
print("\nFolded back shape:", T_recon.shape)
print("Reconstruction exact:", np.allclose(T, T_recon))

### 3.5 Mode-n Product

The **mode-$n$ product** of tensor $\mathcal{T} \in \mathbb{R}^{I_1 \times \cdots \times I_N}$ with matrix $U \in \mathbb{R}^{J \times I_n}$:

$$(\mathcal{T} \times_n U)_{i_1 \cdots i_{n-1}\; j\; i_{n+1} \cdots i_N} = \sum_{i_n} \mathcal{T}_{i_1 \cdots i_n \cdots i_N}\, U_{j i_n}$$

This is used extensively in Tucker decompositions — it applies a matrix transformation along one mode.

In [ ]:
T = np.random.randn(4, 5, 6)   # shape (I1=4, I2=5, I3=6)
U = np.random.randn(3, 5)      # contract mode-1 (size 5) down to size 3

result = tl.tenalg.mode_dot(T, U, mode=1)
print("T shape:", T.shape)
print("U shape:", U.shape)
print("T x_1 U shape:", result.shape)   # expect (4, 3, 6)

# Manually verify one element
manual = sum(T[0, j, 0] * U[0, j] for j in range(5))
print(f"Manual result[0,0,0] = {manual:.6f}")
print(f"tenalg result[0,0,0] = {result[0,0,0]:.6f}")
print("Match:", np.isclose(manual, result[0,0,0]))

## 4. The Frobenius Norm

The most common norm for tensors is the **Frobenius norm** — just square all elements, sum, take the square root:

$$\| \mathcal{T} \|_F = \sqrt{\sum_{i_1, \ldots, i_N} T_{i_1 \cdots i_N}^2}$$

It is used to measure reconstruction error in decompositions.

In [ ]:
T = np.random.randn(3, 4, 5)

frob = np.linalg.norm(T)   # works for any shape
manual_frob = np.sqrt(np.sum(T**2))

print(f"np.linalg.norm: {frob:.6f}")
print(f"Manual Frobenius: {manual_frob:.6f}")
print("Match:", np.isclose(frob, manual_frob))

## 5. Einstein Summation (`einsum`) — The Universal Tool

`np.einsum` is the backbone of almost all tensor operations.  
The string `'ij,jk->ik'` means: "sum over shared index $j$, keep $i$ and $k$".

| Operation | einsum string |
|-----------|---------------|
| dot product | `'i,i->'` |
| outer product | `'i,j->ij'` |
| matrix multiply | `'ij,jk->ik'` |
| trace | `'ii->'` |
| batch matmul | `'bij,bjk->bik'` |
| rank-3 contraction | `'ij,jkl->ikl'` |

In [ ]:
import opt_einsum as oe

A = np.random.randn(10, 20)
B = np.random.randn(20, 30)
C = np.random.randn(30, 5)

# Chain of matrix multiplications — opt_einsum finds the cheapest order
result_numpy  = np.einsum('ij,jk,kl->il', A, B, C)
result_opt    = oe.contract('ij,jk,kl->il', A, B, C)

print("Result shape:", result_opt.shape)
print("numpy vs opt_einsum match:", np.allclose(result_numpy, result_opt))

# Show the optimal contraction path found
path, info = oe.contract_path('ij,jk,kl->il', A, B, C)
print("Optimal contraction path:", path)
print(info)

## Summary

| Concept | Key Idea |
|---------|----------|
| Tensor | Multi-dimensional array; order = number of indices |
| Index notation | $T_{ijk}$ labels individual elements |
| Contraction | Sum over a shared index (generalises matrix multiply) |
| Unfolding | Reshape tensor into matrix; needed for SVD-based methods |
| Mode-n product | Apply matrix transformation along one mode |
| Frobenius norm | Measure size/error for tensors |
| einsum | Compact notation for any tensor operation |

You now have the vocabulary for the rest of the course. ➡️ **02_decompositions.ipynb**